# GPU Solution — CFPB Consumer Complaints

**Course:** Engineering of Data Analysis (2779), T4 2025-26, NOVA SBE

Runs on **Colab GPU (T4) runtime** only. Loads best hyperparams from `results/best_params.json`
written by CPU_Baseline.ipynb, runs a GPU XGBoost CV for context timing, then a
3 × 5 fraction GPU-only sweep. Writes `results/timings_gpu.csv`.

**Runtime:** T4 GPU. Go to *Runtime → Change runtime type → T4 GPU* before running.


## 1. RAPIDS Installation

Install cuDF and cuML (CUDA 12 wheels). Run once, then **restart the runtime** before proceeding.

In [ ]:
import subprocess
gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if 'NVIDIA' not in gpu_info.stdout:
    raise RuntimeError('No GPU detected — go to Runtime > Change runtime type > T4 GPU')
print('GPU OK:', [l for l in gpu_info.stdout.split('\n') if 'MiB' in l or 'T4' in l][:2])

!pip install --extra-index-url=https://pypi.nvidia.com cudf-cu12==25.02.* cuml-cu12==25.02.* -q
print('\nInstallation complete. RESTART THE RUNTIME now, then skip this cell.')


## 2. Imports & GPU Warm-up

In [ ]:
try:
    import cudf
    import cuml
except ImportError:
    raise RuntimeError('cuDF/cuML not found — run the install cell above and restart the runtime.')

import numpy as np
import pandas as pd
import cudf
import cuml
from cuml.linear_model import LogisticRegression as cuLR

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

import time
import json
import os
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print(f'cuDF  {cudf.__version__}  |  cuML  {cuml.__version__}')

# Force CUDA context initialisation before any timing
_warmup = cudf.Series([1.0, 2.0, 3.0]).sum()
print('GPU warm-up done.')


## 3. Data Loading

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust if your file is in a subfolder
DATA_PATH = '/content/drive/MyDrive/complaints_training.csv'

df_raw = pd.read_csv(DATA_PATH)
print(f'Loaded {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')


## 4. Train/Test Split

Identical seed to CPU_Baseline.ipynb — same subsets.

In [ ]:
X = df_raw.drop(columns=['Complaint ID', 'Consumer disputed?'])
y = df_raw['Consumer disputed?'].map({'Yes': 1, 'No': 0})

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f'Train: {len(X_train_raw):,}  Test: {len(X_test_raw):,}')


## 5. Feature Engineering

### 5.1 CPU Reference Function (not benchmarked)

Used only for the correctness comparison in 5.3.

In [ ]:
def feature_engineering(X, y=None, fit_encoders=None):
    encoders = {} if fit_encoders is None else fit_encoders
    training = fit_encoders is None
    raw = X.copy()
    for col in ['Complaint ID', 'Consumer disputed?']:
        if col in raw.columns:
            raw = raw.drop(columns=[col])
    out = pd.DataFrame(index=raw.index)

    date_rec  = pd.to_datetime(raw['Date received'],        errors='coerce')
    date_sent = pd.to_datetime(raw['Date sent to company'], errors='coerce')
    out['response_lag_days'] = (date_sent - date_rec).dt.days.clip(0, 120).fillna(0).astype(int)
    out['received_year']     = date_rec.dt.year.fillna(2015).astype(int)
    out['day_of_week']       = date_rec.dt.dayofweek.fillna(0).astype(int)

    resp = raw['Company response to consumer'].fillna('Missing')
    out['company_response_monetary']       = resp.str.contains('monetary',       case=False).astype(int)
    out['company_response_non_monetary']   = resp.str.contains('non-monetary',   case=False).astype(int)
    out['company_response_explanation']    = resp.str.contains('explanation',    case=False).astype(int)
    out['company_response_without_relief'] = resp.str.contains('without relief', case=False).astype(int)
    out['company_response_in_progress']    = resp.str.contains('in progress',    case=False).astype(int)
    out['company_response_is_relief']      = (
        (out['company_response_monetary'] == 1) |
        (out['company_response_non_monetary'] == 1)
    ).astype(int)
    out['timely_response']     = raw['Timely response?'].fillna('No').eq('Yes').astype(int)
    out['closed_x_timely']     = out['company_response_is_relief'] * out['timely_response']
    out['has_public_response'] = raw['Company public response'].notna().astype(int)

    narr = raw['Consumer complaint narrative'].fillna('')
    out['has_narrative']       = (narr != '').astype(int)
    out['word_count']          = narr.apply(lambda x: len(x.split()) if x else 0)
    out['char_count']          = narr.str.len().fillna(0).astype(int)
    out['avg_word_length']     = narr.apply(
        lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0
    )
    out['narrative_short']     = (out['word_count'].between(1, 19)).astype(int)
    out['narrative_long']      = (out['word_count'] > 200).astype(int)
    esc = 'attorney|lawyer|lawsuit|legal action|sue|court|cfpb|regulation|violation|fraud|illegal|discriminat'
    out['escalation_term_count'] = narr.str.lower().str.count(esc).fillna(0).astype(int)
    out['has_escalation_terms']  = (out['escalation_term_count'] > 0).astype(int)
    out['narrative_x_no_relief'] = out['has_narrative'] * (1 - out['company_response_is_relief'])

    consent = raw['Consumer consent provided?'].fillna('Missing')
    out['consent_provided']  = consent.eq('Consent provided').astype(int)
    out['consent_withdrawn'] = consent.eq('Consent withdrawn').astype(int)
    out['has_subproduct']    = raw['Sub-product'].notna().astype(int)
    out['has_subissue']      = raw['Sub-issue'].notna().astype(int)
    tags = raw['Tags'].fillna('')
    out['tags_any']          = (tags != '').astype(int)
    out['is_older_american'] = tags.str.contains('Older American', case=False).astype(int)
    out['is_servicemember']  = tags.str.contains('Servicemember',  case=False).astype(int)

    def target_encode(series, name, smoothing=20):
        if training:
            tmp = pd.DataFrame({'val': series.values, 'target': y.values}, index=series.index)
            agg = tmp.groupby('val')['target'].agg(['mean', 'count'])
            global_mean = y.mean()
            smooth = (agg['count'] * agg['mean'] + smoothing * global_mean) / (agg['count'] + smoothing)
            encoders[name] = {'map': smooth.to_dict(), 'global_mean': global_mean}
        return series.map(encoders[name]['map']).fillna(encoders[name]['global_mean'])

    def freq_encode(series, name):
        if training:
            encoders[name] = series.value_counts().to_dict()
        return series.map(encoders[name]).fillna(1)

    out['Product_te']           = target_encode(raw['Product'].fillna('Missing'),                      'Product_te')
    out['Issue_te']             = target_encode(raw['Issue'].fillna('Missing'),                        'Issue_te')
    out['State_te']             = target_encode(raw['State'].fillna('Missing'),                        'State_te')
    out['Company_te']           = target_encode(raw['Company'].fillna('Missing'),                      'Company_te')
    out['Sub_product_te']       = target_encode(raw['Sub-product'].fillna('Missing'),                  'Sub_product_te')
    out['Sub_issue_te']         = target_encode(raw['Sub-issue'].fillna('Missing'),                    'Sub_issue_te')
    out['Submitted_via_te']     = target_encode(raw['Submitted via'].fillna('Missing'),                'Submitted_via_te')
    out['company_response_te']  = target_encode(raw['Company response to consumer'].fillna('Missing'), 'company_response_te')
    out['zip3_region_te']       = target_encode(
        raw['ZIP code'].fillna('000').astype(str).str[:3],                                             'zip3_region_te')
    prod_x_channel = raw['Product'].fillna('Missing') + '_' + raw['Submitted via'].fillna('Missing')
    out['product_x_channel_te'] = target_encode(prod_x_channel, 'product_x_channel_te')
    prod_x_issue = raw['Product'].fillna('Missing') + '_' + raw['Issue'].fillna('Missing')
    out['product_issue_te']     = target_encode(prod_x_issue, 'product_issue_te')
    out['company_volume']       = freq_encode(raw['Company'].fillna('Missing'), 'company_volume')
    out['response_te_x_relief'] = out['company_response_te'] * out['company_response_is_relief']
    return out, encoders


### 5.2 GPU Feature Engineering (`feature_engineering_gpu`)

Three changes from the CPU version: (1) `apply()` lambdas replaced with cuDF vectorised ops, (2) RE2-safe regex, (3) target encoding via cuDF groupby.

In [ ]:
def feature_engineering_gpu(X, y=None, fit_encoders=None):
    encoders = {} if fit_encoders is None else fit_encoders
    training = fit_encoders is None

    raw = cudf.DataFrame.from_pandas(X.copy().reset_index(drop=True))
    if training:
        y_gpu = cudf.Series(y.values)

    for col in ['Complaint ID', 'Consumer disputed?']:
        if col in raw.columns:
            raw = raw.drop(columns=[col])

    out = cudf.DataFrame()

    date_rec  = cudf.to_datetime(raw['Date received'],        errors='coerce')
    date_sent = cudf.to_datetime(raw['Date sent to company'], errors='coerce')
    lag = (date_sent - date_rec).dt.days
    out['response_lag_days'] = lag.clip(0, 120).fillna(0).astype('int32')
    out['received_year']     = date_rec.dt.year.fillna(2015).astype('int32')
    out['day_of_week']       = date_rec.dt.dayofweek.fillna(0).astype('int32')

    resp = raw['Company response to consumer'].fillna('Missing')
    out['company_response_monetary']       = resp.str.contains('monetary',       regex=False).astype('int32')
    out['company_response_non_monetary']   = resp.str.contains('non-monetary',   regex=False).astype('int32')
    out['company_response_explanation']    = resp.str.contains('explanation',    regex=False).astype('int32')
    out['company_response_without_relief'] = resp.str.contains('without relief', regex=False).astype('int32')
    out['company_response_in_progress']    = resp.str.contains('in progress',    regex=False).astype('int32')
    out['company_response_is_relief']      = (
        (out['company_response_monetary'] == 1) |
        (out['company_response_non_monetary'] == 1)
    ).astype('int32')
    out['timely_response']     = (raw['Timely response?'].fillna('No') == 'Yes').astype('int32')
    out['closed_x_timely']     = out['company_response_is_relief'] * out['timely_response']
    out['has_public_response'] = (~raw['Company public response'].isna()).astype('int32')

    narr = raw['Consumer complaint narrative'].fillna('')
    out['has_narrative'] = (narr != '').astype('int32')
    out['word_count']    = narr.str.token_count().astype('int32')
    out['char_count']    = narr.str.len().fillna(0).astype('int32')
    wc = out['word_count'].clip(lower=1)
    out['avg_word_length'] = ((out['char_count'] - wc + 1) / wc).fillna(0).clip(lower=0)
    out['narrative_short'] = (out['word_count'].between(1, 19)).astype('int32')
    out['narrative_long']  = (out['word_count'] > 200).astype('int32')
    esc = ('attorney|lawyer|lawsuit|legal action|sue|court|cfpb|'
           'regulation|violation|fraud|illegal|discriminat')
    narr_lower = narr.str.lower()
    out['escalation_term_count'] = narr_lower.str.count(esc).fillna(0).astype('int32')
    out['has_escalation_terms']  = (out['escalation_term_count'] > 0).astype('int32')
    out['narrative_x_no_relief'] = out['has_narrative'] * (1 - out['company_response_is_relief'])

    consent = raw['Consumer consent provided?'].fillna('Missing')
    out['consent_provided']  = (consent == 'Consent provided').astype('int32')
    out['consent_withdrawn'] = (consent == 'Consent withdrawn').astype('int32')
    out['has_subproduct']    = (~raw['Sub-product'].isna()).astype('int32')
    out['has_subissue']      = (~raw['Sub-issue'].isna()).astype('int32')
    tags = raw['Tags'].fillna('')
    out['tags_any']          = (tags != '').astype('int32')
    out['is_older_american'] = tags.str.contains('Older American', regex=False).astype('int32')
    out['is_servicemember']  = tags.str.contains('Servicemember',  regex=False).astype('int32')

    def target_encode_gpu(series, name, smoothing=20):
        series = series.reset_index(drop=True)
        if training:
            tmp = cudf.DataFrame({'val': series, 'target': y_gpu})
            agg = tmp.groupby('val')['target'].agg(['mean', 'count']).reset_index()
            agg.columns = ['val', 'mean', 'count']
            g = float(y_gpu.mean())
            agg['smooth'] = (agg['count'] * agg['mean'] + smoothing * g) / (agg['count'] + smoothing)
            encoders[name] = {
                'map': dict(zip(agg['val'].to_pandas(), agg['smooth'].to_pandas())),
                'global_mean': g
            }
        return series.map(encoders[name]['map']).fillna(encoders[name]['global_mean'])

    def freq_encode_gpu(series, name):
        series = series.reset_index(drop=True)
        if training:
            vc = series.value_counts()
            encoders[name] = dict(zip(vc.index.to_pandas(), vc.to_pandas()))
        return series.map(encoders[name]).fillna(1)

    out['Product_te']           = target_encode_gpu(raw['Product'].fillna('Missing'),                      'Product_te')
    out['Issue_te']             = target_encode_gpu(raw['Issue'].fillna('Missing'),                        'Issue_te')
    out['State_te']             = target_encode_gpu(raw['State'].fillna('Missing'),                        'State_te')
    out['Company_te']           = target_encode_gpu(raw['Company'].fillna('Missing'),                      'Company_te')
    out['Sub_product_te']       = target_encode_gpu(raw['Sub-product'].fillna('Missing'),                  'Sub_product_te')
    out['Sub_issue_te']         = target_encode_gpu(raw['Sub-issue'].fillna('Missing'),                    'Sub_issue_te')
    out['Submitted_via_te']     = target_encode_gpu(raw['Submitted via'].fillna('Missing'),                'Submitted_via_te')
    out['company_response_te']  = target_encode_gpu(raw['Company response to consumer'].fillna('Missing'), 'company_response_te')
    out['zip3_region_te']       = target_encode_gpu(
        raw['ZIP code'].fillna('000').astype(str).str.slice(0, 3),                                         'zip3_region_te')
    prod_x_channel = raw['Product'].fillna('Missing') + '_' + raw['Submitted via'].fillna('Missing')
    out['product_x_channel_te'] = target_encode_gpu(prod_x_channel, 'product_x_channel_te')
    prod_x_issue = raw['Product'].fillna('Missing') + '_' + raw['Issue'].fillna('Missing')
    out['product_issue_te']     = target_encode_gpu(prod_x_issue, 'product_issue_te')
    out['company_volume']       = freq_encode_gpu(raw['Company'].fillna('Missing'), 'company_volume')
    out['response_te_x_relief'] = out['company_response_te'] * out['company_response_is_relief']
    return out.to_pandas(), encoders


### 5.3 Correctness Verification

Run GPU FE on the full training set (result is reused for the XGB CV below). Then compare with CPU FE to verify numerical equivalence within tolerances.

In [ ]:
# GPU FE on full training set — result kept for XGB CV
X_train_enc_gpu, enc_gpu = feature_engineering_gpu(X_train_raw, y=y_train)
X_test_enc_gpu, _        = feature_engineering_gpu(X_test_raw, fit_encoders=enc_gpu)

# CPU FE for comparison only
X_train_enc_cpu, _ = feature_engineering(X_train_raw, y=y_train)

# Verify within tolerances
cols_approx = {'avg_word_length'}
max_diff_exact  = 0.0
max_diff_approx = 0.0

for col in X_train_enc_cpu.columns:
    cpu_v = X_train_enc_cpu[col].values.astype(float)
    gpu_v = X_train_enc_gpu[col].values.astype(float)
    rel = float(np.abs(cpu_v - gpu_v).max() / (np.abs(cpu_v).max() + 1e-9))
    if col in cols_approx:
        max_diff_approx = max(max_diff_approx, rel)
    else:
        max_diff_exact = max(max_diff_exact, rel)

print(f'Max rel diff (exact cols):      {max_diff_exact:.2e}  (threshold: 1e-4)')
print(f'Max rel diff (avg_word_length): {max_diff_approx:.4f}  (threshold: 0.02)')
assert max_diff_exact  < 1e-4, f'Exact column divergence: {max_diff_exact}'
assert max_diff_approx < 0.02, f'avg_word_length divergence: {max_diff_approx}'
print('Correctness check PASSED.')
print(f'GPU FE shape: {X_train_enc_gpu.shape}')
del X_train_enc_cpu  # free memory


## 6. Load Best Hyperparameters

Loaded from `results/best_params.json` written by CPU_Baseline.ipynb.
Both CPU and GPU sweeps use these same params — ensures apples-to-apples timing comparison.

In [ ]:
with open('results/best_params.json') as f:
    best_params = json.load(f)

print('Best LR params: ', best_params['lr'])
print('Best XGB params:', best_params['xgb'])
print('CPU AUC — LR:', best_params['auc']['lr'],  ' XGB:', best_params['auc']['xgb'])
print('CPU CV context — LR:', round(best_params['cv_context_seconds']['lr_gridsearch'], 1), 's',
      ' XGB:', round(best_params['cv_context_seconds']['xgb_randomizedsearch'], 1), 's')


## 7. GPU XGBoost CV (Context Timing)

One full RandomizedSearchCV run on the GPU. Result is a **context timing** only — the best params from the CPU run are used in the sweep, not these GPU params.

In [ ]:
xgb_pipeline_gpu = Pipeline([
    ('imputer',    SimpleImputer(strategy='median')),
    ('classifier', XGBClassifier(
        random_state=42, eval_metric='logloss',
        tree_method='hist', device='cuda'
    ))
])

param_grid_xgb = {
    'classifier__n_estimators':     [400, 500, 600, 800],
    'classifier__max_depth':        [4, 5, 6],
    'classifier__learning_rate':    [0.02, 0.03, 0.05],
    'classifier__scale_pos_weight': [3.5, 4, 4.5],
    'classifier__min_child_weight': [5, 10, 15],
    'classifier__subsample':        [0.55, 0.6, 0.65, 0.7],
    'classifier__colsample_bytree': [0.6, 0.65, 0.7],
    'classifier__reg_alpha':        [0, 0.1, 1],
    'classifier__reg_lambda':       [3, 5, 7]
}

cv_xgb = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search_xgb_gpu = RandomizedSearchCV(
    xgb_pipeline_gpu, param_grid_xgb,
    n_iter=60, cv=cv_xgb, scoring='f1',
    random_state=42, n_jobs=1, verbose=1
)

t0 = time.perf_counter()
search_xgb_gpu.fit(X_train_enc_gpu, y_train)
t_xgb_cv_gpu = time.perf_counter() - t0

y_prob_xgb_gpu = search_xgb_gpu.best_estimator_.predict_proba(X_test_enc_gpu)[:, 1]
auc_xgb_gpu    = roc_auc_score(y_test, y_prob_xgb_gpu)

auc_diff = abs(auc_xgb_gpu - best_params['auc']['xgb'])
print(f'[CONTEXT] GPU XGB RandomizedCV (300 fits): {t_xgb_cv_gpu:.1f}s  ({t_xgb_cv_gpu/60:.1f} min)')
print(f'Test ROC-AUC: {auc_xgb_gpu:.4f}  |  AUC diff vs CPU: {auc_diff:.4f}  (threshold: 0.005)')
assert auc_diff < 0.005, f'XGB AUC divergence too large: {auc_diff:.4f}'
print('XGB AUC correctness check PASSED.')


## 8. GPU Warm-up Before Sweep

Force JIT compilation and memory allocation before timed iterations begin.

In [ ]:
print('Warm-up: running throwaway GPU ops...')

_Xw = X_train_raw.sample(n=3000, random_state=0)
_yw = y_train.loc[_Xw.index]
_Xw_enc, _ = feature_engineering_gpu(_Xw, y=_yw)

# cuML LR warm-up
_imp = SimpleImputer(strategy='median'); _scl = StandardScaler()
_Xw_pp = _scl.fit_transform(_imp.fit_transform(_Xw_enc)).astype(np.float32)
_sw = compute_sample_weight('balanced', _yw.values).astype(np.float32)
_lr_w = cuLR(C=0.1, penalty='l2', max_iter=50)
_lr_w.fit(_Xw_pp, _yw.values.astype(np.float32), sample_weight=_sw)
_ = _lr_w.predict_proba(_Xw_pp)

# XGB GPU warm-up
_xgb_w = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('clf', XGBClassifier(n_estimators=10, tree_method='hist', device='cuda',
                          eval_metric='logloss', random_state=42))
])
_xgb_w.fit(_Xw_enc, _yw)
_ = _xgb_w.predict_proba(_Xw_enc)

print('Warm-up complete. Timing starts now.')


## 9. GPU-Only Sweep

3 runs × 5 fractions × 6 phases = 90 rows. Same structure as CPU_Baseline.ipynb.

In [ ]:
FRACS  = [0.10, 0.25, 0.50, 0.75, 1.00]
RUNS   = 3
DEVICE = 'gpu'

best_lr_C       = best_params['lr']['C']
best_lr_penalty = best_params['lr'].get('penalty', 'l2')
best_xgb_p      = {k: v for k, v in best_params['xgb'].items()}

os.makedirs('results', exist_ok=True)
_rows = []

def log_row(device, fraction, n_rows, phase, run_idx, seconds, notes=''):
    _rows.append({
        'device': device, 'fraction': fraction, 'n_rows': n_rows,
        'phase': phase, 'run_idx': run_idx, 'seconds': seconds, 'notes': notes
    })

for run_idx in range(RUNS):
    for frac in FRACS:
        X_sub = X_train_raw.sample(frac=frac, random_state=42)
        y_sub = y_train.loc[X_sub.index]
        n = len(X_sub)

        # fe_train (GPU)
        t0 = time.perf_counter()
        X_sub_enc, enc_sub = feature_engineering_gpu(X_sub, y=y_sub)
        log_row(DEVICE, frac, n, 'fe_train', run_idx, time.perf_counter() - t0)

        # fe_test (fraction encoders, GPU)
        t0 = time.perf_counter()
        X_test_sub, _ = feature_engineering_gpu(X_test_raw, fit_encoders=enc_sub)
        log_row(DEVICE, frac, n, 'fe_test', run_idx, time.perf_counter() - t0)

        # lr_fit (cuML — includes sklearn imputer + scaler to match CPU pipeline structure)
        t0 = time.perf_counter()
        imp = SimpleImputer(strategy='median')
        scl = StandardScaler()
        X_sub_pp = scl.fit_transform(imp.fit_transform(X_sub_enc)).astype(np.float32)
        sw = compute_sample_weight('balanced', y_sub.values).astype(np.float32)
        lr_gpu = cuLR(C=best_lr_C, penalty=best_lr_penalty, max_iter=1000)
        lr_gpu.fit(X_sub_pp, y_sub.values.astype(np.float32), sample_weight=sw)
        log_row(DEVICE, frac, n, 'lr_fit', run_idx, time.perf_counter() - t0)

        # lr_predict (cuML — includes test preprocessing to match CPU)
        t0 = time.perf_counter()
        X_test_pp = scl.transform(imp.transform(X_test_sub)).astype(np.float32)
        lr_gpu.predict_proba(X_test_pp)
        log_row(DEVICE, frac, n, 'lr_predict', run_idx, time.perf_counter() - t0)

        # xgb_fit (GPU, Pipeline includes imputer)
        xgb_gpu = Pipeline([
            ('imputer',    SimpleImputer(strategy='median')),
            ('classifier', XGBClassifier(
                **best_xgb_p,
                tree_method='hist', device='cuda',
                random_state=42, eval_metric='logloss'
            ))
        ])
        t0 = time.perf_counter()
        xgb_gpu.fit(X_sub_enc, y_sub)
        log_row(DEVICE, frac, n, 'xgb_fit', run_idx, time.perf_counter() - t0)

        # xgb_predict (GPU)
        t0 = time.perf_counter()
        xgb_gpu.predict_proba(X_test_sub)
        log_row(DEVICE, frac, n, 'xgb_predict', run_idx, time.perf_counter() - t0)

        print(f'run={run_idx} frac={frac:.0%} n={n:,} done')

    # Save after every outer run
    pd.DataFrame(_rows).to_csv('results/timings_gpu.csv', index=False)
    print(f'[run {run_idx}] Saved results/timings_gpu.csv  ({len(_rows)} rows so far)')


In [ ]:
df_gpu = pd.read_csv('results/timings_gpu.csv')
expected = RUNS * len(FRACS) * 6
print(f'Total rows: {len(df_gpu)}  (expected {expected})')
assert len(df_gpu) == expected, f'Row count mismatch: got {len(df_gpu)}'
print('Row count OK.')
print()
print(df_gpu.groupby(['phase', 'fraction'])['seconds'].median().unstack().to_string())
